# Outlook

In this notebook, we will implement the REINFORCE algorithm and several of
its variants (discounted rewards, learned baselines).

The REINFORCE algorithm is explained in a series of 3 videos: [video
1](https://www.youtube.com/watch?v=R7ULMBXOQtE), [video
2](https://www.youtube.com/watch?v=dKUWto9B9WY) and [video
3](https://www.youtube.com/watch?v=GcJ9hl3T6x8). You can also read the
corresponding slides:
[slides1](http://pages.isir.upmc.fr/~sigaud/teach/ps/3_pg_derivation1.pdf),
[slides2](http://pages.isir.upmc.fr/~sigaud/teach/ps/4_pg_derivation2.pdf),
[slides3](http://pages.isir.upmc.fr/~sigaud/teach/ps/5_pg_derivation3.pdf).

## Setup

**On Colab**: run the installation cell below (generated automatically,
pinned versions). **Locally**: install the environment with `uv sync`, no
need for this cell.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, replace
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.profiles import Profile
from rl_mind.core import Actor, StochasticAction
from rl_mind.nn import build_mlp
from rl_mind.env import VecEnv
from rl_mind.data import Episode
from rl_mind.collectors import EpisodeCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


## The `rl_mind` toolkit

The `rl_mind` package ([documentation](https://master-dac.isir.upmc.fr/rl/rl-mind/)),
installed with the practicals, provides a small set of *typed* helpers used
throughout the RL notebooks. The ones used here:

- `VecEnv` runs several copies of a gymnasium environment in parallel,
  using torch tensors;
- `Episode` stores one full episode: `obs` (`[T, obs_dim]`), `action`
  (tensors of shape `[T, ...]`), `reward` (`[T]`), the last observation
  `final_obs` and whether the episode `terminated` (as opposed to being
  truncated by a time limit); `all_obs` (`[T+1, obs_dim]`) is `obs` with
  `final_obs` appended, i.e. the observations $s_0, \ldots, s_T$;
- `EpisodeCollector` runs an actor in a `VecEnv` and returns complete
  `Episode`s;
- `Evaluator` periodically evaluates an actor and keeps a copy of the best
  one;
- `Actor` is the base class of all the policies: `actor(obs)` returns an
  `Action` (its training-time, stochastic behavior) while `actor.act(obs)`
  returns the action tensor of its deterministic (evaluation) behavior.

In [ ]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

# REINFORCE

[REINFORCE](https://link.springer.com/content/pdf/10.1007/BF00992696.pdf)
is an *episodic*, *on-policy* policy-gradient algorithm: it collects full
episodes with the current **stochastic** policy, and follows the gradient

$$
\nabla_\theta \mathbb{E}_\tau[R(\tau)]
\approx
\frac{1}{m} \sum_{i=1}^m \sum_{t=0}^{H_i-1} A(s_t^{(i)})\,
\nabla_\theta \log \pi_\theta(a_t^{(i)} | s_t^{(i)})
$$

with $m$ the number of episodes used to estimate the gradient, $H_i$ the
length of episode $i$, and $A$ an *advantage* function whose different
flavors (summed rewards, discounted rewards, with or without a baseline)
are the subject of this notebook.

## Definition of the agents

### The policy

The policy is a neural network that maps a state to a **distribution**
over the (discrete) actions, using
[`torch.distributions`](https://pytorch.org/docs/stable/distributions.html).
During training (`forward`), the action is *sampled* from the
distribution; during evaluation (`act`), we take the most probable action.

Note that the returned `StochasticAction` also stores the log-probability
of the sampled action.

In [ ]:
class DiscretePolicy(Actor[StochasticAction]):
    """A stochastic policy for discrete actions, $\\pi(a|s)$"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...], n_actions: int):
        super().__init__()
        self.model = build_mlp([obs_dim, *hidden, n_actions])

    def dist(self, obs: Tensor) -> torch.distributions.Categorical:
        """The distribution over actions for a batch of observations"""
        return torch.distributions.Categorical(logits=self.model(obs))

    def forward(self, obs: Tensor) -> StochasticAction:
        dist = self.dist(obs)
        action = dist.sample()
        return StochasticAction(value=action, log_prob=dist.log_prob(action))

    def act(self, obs: Tensor) -> Tensor:
        return self.model(obs).argmax(-1)

### The critic $V(s)$

The value network takes an observation as input and outputs the value
$V(s)$ of this observation. It will be used as a *baseline* to reduce the
variance of the gradient estimate.

In [ ]:
class ValueNetwork(nn.Module):
    """The value network $V(s)$"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...]):
        super().__init__()
        self.model = build_mlp([obs_dim, *hidden, 1])

    def forward(self, obs: Tensor) -> Tensor:
        """Compute $V(s)$ for a batch: `[B, obs_dim] -> [B]`"""
        return self.model(obs).squeeze(-1)

## Hyper-parameters

In [ ]:
@dataclass(frozen=True)
class ReinforceConfig:
    env_name: str = "CartPole-v1"
    seed: int = 1

    #: Number of gradient updates
    max_updates: int = Profile.pick(700, fast_test=15)
    #: Number of parallel training environments
    n_envs: int = 8
    #: Number of episodes collected for each update
    episodes_per_update: int = 8

    #: Discount factor
    gamma: float = 0.99
    #: Weight of the critic loss (when a baseline is learned)
    critic_coef: float = 1.0

    actor_hidden: tuple[int, ...] = (32,)
    critic_hidden: tuple[int, ...] = (36,)
    lr: float = 1e-3

    #: Steps between two evaluations, and number of evaluation episodes
    eval_interval: int = Profile.pick(2_000, fast_test=200)
    n_eval_envs: int = 10

## Main training loop

The training loop is parametrized by the two functions that define the
REINFORCE flavor:

- `compute_advantage(cfg, episode, values)` computes the advantage
  $A(s_t)$ at each step of an episode (a tensor of shape `[T]`). `values`
  contains the critic values $V(s_0), \ldots, V(s_T)$ — a tensor of shape
  `[T+1]` that *includes the final observation* — and can be ignored by
  the variants without a baseline;
- `compute_critic_loss(cfg, episode, values)` computes the loss of the
  critic (`None` when no baseline is learned).

Note that the advantage is `.detach()`-ed in the policy loss: the policy
gradient must not flow through the critic.

In [ ]:
def run_reinforce(
    cfg: ReinforceConfig, variant: str, compute_advantage, compute_critic_loss=None
) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(cfg.env_name, cfg.n_envs, seed=cfg.seed)

    policy = DiscretePolicy(env.observation_dim, cfg.actor_hidden, env.n_actions)
    critic = ValueNetwork(env.observation_dim, cfg.critic_hidden)
    optimizer = torch.optim.Adam(
        [*policy.parameters(), *critic.parameters()], lr=cfg.lr
    )

    collector = EpisodeCollector(env, policy)
    run_dir = run_directory(f"reinforce-{variant}-{cfg.env_name}-S{cfg.seed}")
    writer = SummaryWriter(run_dir)
    evaluator = Evaluator(
        VecEnv(cfg.env_name, cfg.n_eval_envs, seed=cfg.seed + 100),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=writer,
    )

    pbar = tqdm(range(cfg.max_updates))
    for update in pbar:
        episodes = collector.collect(cfg.episodes_per_update)

        step_losses: list[Tensor] = []
        critic_losses: list[Tensor] = []
        for episode in episodes:
            # The values V(s_0) ... V(s_T): `all_obs` is `obs` with the final
            # observation appended, so this vector has T+1 entries
            values = critic(episode.all_obs)

            # Compute the policy loss terms (one value per step) for this episode

            # Compute the advantage...
            advantage = ...
            # ... and the log-probability of the actions that were played
            # (`policy.dist(...)` gives the action distribution at each step)
            log_probs = ...
            step_losses.append(...)
            assert False, 'Not implemented yet'


            if compute_critic_loss is not None:
                critic_losses.append(compute_critic_loss(cfg, episode, values))

        # Average over all the steps of all the episodes
        loss = torch.cat(step_losses).mean()
        writer.add_scalar("loss/actor", loss.item(), collector.steps)
        if critic_losses:
            critic_loss = torch.stack(critic_losses).mean()
            writer.add_scalar("loss/critic", critic_loss.item(), collector.steps)
            loss = loss + cfg.critic_coef * critic_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if result := evaluator.run_if_needed(collector.steps, policy):
            pbar.set_description(
                f"eval={result.mean:6.1f} best={evaluator.best_reward:6.1f}"
            )

    return evaluator

## Launching tensorboard to visualize the results

In [ ]:
setup_tensorboard()

In [ ]:
params = ReinforceConfig()

# First variant: summing all the rewards of the episode

The most basic variant of the policy-gradient algorithms uses, at every
step, the sum of *all* the rewards of the episode:
$A(s_t) = \sum_{k=1}^{H} r_k$.

This is implemented by the `sum_of_rewards` function below.

In [ ]:
def sum_of_rewards(cfg: ReinforceConfig, episode: Episode, values: Tensor) -> Tensor:
    """The advantage at each step is the total reward of the episode"""
    return torch.full_like(episode.reward, episode.cumulated_reward)

In [ ]:
reinforce_sum = run_reinforce(params, "sum", sum_of_rewards)

In [ ]:
reinforce_sum.visualize_best()

# Exercises

## Second variant: summing the discounted rewards to go

As explained in the [second
video](https://www.youtube.com/watch?v=dKUWto9B9WY) and [the corresponding
slides](http://pages.isir.upmc.fr/~sigaud/teach/ps/4_pg_derivation2.pdf),
using the *discounted* rewards *after* the current step (and ignoring the
rewards before it) results in a lower variance:
$A(s_t) = G_t = \sum_{k=t}^{H-1} \gamma^{k-t}\, r_{k+1}$.

By taking inspiration from the `sum_of_rewards` function above, code a
`discounted_rewards_to_go` function.

Hint: proceed *backwards*, starting from the last step of the episode
($G_{H-1} = r_H$, then $G_t = r_{t+1} + \gamma G_{t+1}$).

In [ ]:
def discounted_rewards_to_go(
    cfg: ReinforceConfig, episode: Episode, values: Tensor
) -> Tensor:
    """The advantage at each step is the discounted sum of future rewards $G_t$"""
    # Implement the function (ignore `values`)

    assert False, 'Not implemented yet'


In [ ]:
reinforce_discounted = run_reinforce(params, "discounted", discounted_rewards_to_go)

In [ ]:
reinforce_discounted.visualize_best()

## Third variant: baseline learned with temporal differences

We now reduce the variance further by subtracting a *baseline* from the
rewards to go: the value $V(s_t)$ estimated by the critic,
$A(s_t) = G_t - V(s_t)$.

Code the `discounted_rewards_minus_baseline` function (remember that
`values` has shape `[T+1]`: it includes the value of the final
observation).

In [ ]:
def discounted_rewards_minus_baseline(
    cfg: ReinforceConfig, episode: Episode, values: Tensor
) -> Tensor:
    # Implement the function

    assert False, 'Not implemented yet'


The critic itself is learned simultaneously, here with **temporal
differences**: $V(s_t)$ is regressed towards the target
$r_{t+1} + \gamma V(s_{t+1})$.

Two important details:

- The target is computed with `values[1:].detach()`: we want to apply
  gradient descent on $V(s_t)$ towards the target, *not* to move the
  target itself. In practice, `x.detach()` detaches a tensor from the
  computation graph, so no gradient flows through it.
- The trick to deal with terminal states: if the final state of the
  episode is terminal, $V(s_T)$ does not make sense (there is no future),
  so its target contribution must be ignored — we multiply the last value
  by 0 in that case. If the episode was *truncated* (time limit), the
  agent could have continued and we keep bootstrapping. This trick is used
  in many RL libraries, e.g. SB3.

Code the `td_critic_loss` function.

In [ ]:
def td_critic_loss(cfg: ReinforceConfig, episode: Episode, values: Tensor) -> Tensor:
    """Mean squared temporal-difference error of the critic over one episode"""
    # Implement the function

    # `bootstrap` is 0 for the last step if the episode terminated, 1 elsewhere
    bootstrap = ...
    target = ...
    assert False, 'Not implemented yet'


In [ ]:
reinforce_td = run_reinforce(
    params, "td", discounted_rewards_minus_baseline, td_critic_loss
)

In [ ]:
reinforce_td.visualize_best()

## Fourth variant: Monte-Carlo baseline

The `td_critic_loss` function above uses the temporal-difference approach
to critic estimation. In this part, we compare it to the **Monte-Carlo**
estimation approach.

As explained in [this video](https://www.youtube.com/watch?v=GcJ9hl3T6x8)
and [these
slides](http://pages.isir.upmc.fr/~sigaud/teach/ps/5_pg_derivation3.pdf),
the MC approach directly regresses $V(s_t)$ towards the discounted rewards
to go:

$$\phi_{j+1} = \mathop{\mathrm{argmin}}_{\phi} \frac{1}{m\times H}
\sum_{i=1}^m \sum_{t=0}^{H-1} \left( G_t^{(i)} -
\hat{V}^\pi_{\phi}(s_t^{(i)}) \right)^2 $$

The inner sum of discounted rewards is exactly what
`discounted_rewards_to_go` computes; the rest is a mean squared error.
Code the `mc_critic_loss` function.

In [ ]:
def mc_critic_loss(cfg: ReinforceConfig, episode: Episode, values: Tensor) -> Tensor:
    """Mean squared Monte-Carlo error of the critic over one episode"""
    # Implement the function

    assert False, 'Not implemented yet'


In [ ]:
reinforce_mc = run_reinforce(
    params, "mc", discounted_rewards_minus_baseline, mc_critic_loss
)

In [ ]:
reinforce_mc.visualize_best()

# Comparing the variants

The `history` of each `Evaluator` contains all the evaluations, which we
can plot to compare the variants.

Most probably, the Monte-Carlo variant will not work as well: initially,
the learned critic is a poor estimate of the true $V(s)$, and regressing
directly towards noisy full returns is a slowly-converging objective.

In [ ]:
def plot_evaluations(**evaluators: Evaluator):
    for name, evaluator in evaluators.items():
        steps = [result.step for result in evaluator.history]
        means = torch.tensor([result.mean for result in evaluator.history])
        stds = torch.tensor(
            [float(result.rewards.std()) for result in evaluator.history]
        )
        (line,) = plt.plot(steps, means, label=name)
        plt.fill_between(
            steps, means - stds, means + stds, alpha=0.2, color=line.get_color()
        )
    plt.xlabel("steps")
    plt.ylabel("cumulated reward")
    plt.legend()
    plt.show()


plot_evaluations(
    sum=reinforce_sum,
    discounted=reinforce_discounted,
    td=reinforce_td,
    mc=reinforce_mc,
)